# E2 — Popularity Group Analysis (SNA-MXH)

**Mục đích:** trả lời góp ý của thầy: chia sách thành nhiều mức popularity để xem nhóm nào được gợi ý nhiều hơn và nhóm nào được gợi ý đúng hơn.  
**Input:** `trn_mat.pkl`, `tst_mat.pkl` và các file `topk_test.npz` đã xuất ở E1 (Most Popular, LightGCN, RLMRec-Con) và E3 (User-only, Item-only).  
**Output:** `experiments/E2/popularity_groups/popularity_groups_k10.csv`, `popularity_groups_k20.csv`, biểu đồ exposure theo nhóm.  
**Không training / không inference / không thay đổi checkpoint hay Top-K cũ.**

| Thành phần | Quy ước |
|---|---|
| Item popularity | `degree(i)` = số user tương tác item i trong Train (nhị phân) |
| Nhóm | Sắp xếp giảm dần theo degree, tie-break bằng item ID, chia 5 nhóm bằng nhau G1–G5. G1 = 20% đầu = Head của E2 |
| Tỷ lệ được gợi ý | Số vị trí Top-K thuộc nhóm / tổng số vị trí Top-K |
| Tỷ lệ test | Số tương tác test thuộc nhóm / tổng tương tác test (mốc so sánh) |
| Precision theo nhóm | Số hit thuộc nhóm / số vị trí Top-K thuộc nhóm |
| Recall theo nhóm | Conditional macro, như Head/Tail Recall ở E2: trung bình trên user có ít nhất 1 item test trong nhóm |
| Độ phủ | Tỷ lệ item trong nhóm được gợi ý ít nhất một lần |
| Seeds | 2023, 2024, 2025; báo cáo mean ± SD (ddof=1) |

## CELL 1 — Libraries, Google Drive và đường dẫn
Chạy trên Google Colab. Nếu chạy local/Jupyter, đổi `ROOT` thành thư mục dự án tương ứng.

In [ ]:
import pickle
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy.sparse as sp
from scipy.stats import spearmanr
from IPython.display import display

try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    print('Not running on Google Colab; using local ROOT if configured.')

ROOT = Path('/content/drive/MyDrive/Data_SNA_Group14')
DATA_DIR = ROOT / 'data/raw/amazon'
PRED_DIR = ROOT / 'experiments/predictions'
E3_DIR = ROOT / 'experiments/E3/runs'
E2_SUMMARY_PATH = ROOT / 'experiments/E2/popularity_bias/E2_model_summary.csv'
OUT_DIR = ROOT / 'experiments/E2/popularity_groups'
OUT_DIR.mkdir(parents=True, exist_ok=True)

K_VALUES = (10, 20)
N_GROUPS = 5
SEEDS = (2023, 2024, 2025)
EXPORTS = {
    'Most Popular': {'-': PRED_DIR / 'most_popular/topk_test.npz'},
    'LightGCN': {s: PRED_DIR / f'lightgcn/seed_{s}/topk_test.npz' for s in SEEDS},
    'User-only': {s: E3_DIR / f'user_only/weight_e1_default/seed_{s}/topk/topk_test.npz' for s in SEEDS},
    'Item-only': {s: E3_DIR / f'item_only/weight_e1_default/seed_{s}/topk/topk_test.npz' for s in SEEDS},
    'RLMRec-Con': {s: PRED_DIR / f'lightgcn_plus/seed_{s}/topk_test.npz' for s in SEEDS},
}
missing = [str(p) for runs in EXPORTS.values() for p in runs.values() if not p.is_file()]
assert not missing, f'Missing Top-K exports: {missing}'
print('All Top-K exports found.')

## CELL 2 — Load Train/Test
Chuyển về CSR nhị phân, giữ nguyên indexing của RLMRec.

In [ ]:
def load_binary_csr(path):
    with open(path, 'rb') as f:
        mat = pickle.load(f)
    csr = mat.tocsr(copy=True) if sp.issparse(mat) else sp.csr_matrix(np.asarray(mat))
    csr.sum_duplicates()
    csr.data = (csr.data != 0).astype(np.int8)
    csr.eliminate_zeros()
    csr.sort_indices()
    return csr

train = load_binary_csr(DATA_DIR / 'trn_mat.pkl')
test = load_binary_csr(DATA_DIR / 'tst_mat.pkl')
n_users, n_items = train.shape
print(f'Users={n_users:,} Items={n_items:,} Train={train.nnz:,} Test={test.nnz:,}')

## CELL 3 — Chia 5 nhóm popularity
G1 có cùng 1.867 item với nhóm Head ở E2 (ceil(0.2 × 9.332)).

In [ ]:
degree = np.asarray(train.sum(axis=0)).ravel()
order = np.lexsort((np.arange(n_items), -degree))   # degree giảm dần, tie-break theo item ID
bounds = np.ceil(np.linspace(0, n_items, N_GROUPS + 1)).astype(int)
group = np.empty(n_items, dtype=int)
for g in range(N_GROUPS):
    group[order[bounds[g]:bounds[g + 1]]] = g
GROUP_NAMES = [f'G{g + 1}' for g in range(N_GROUPS)]

group_info = pd.DataFrame({
    'group': GROUP_NAMES,
    'n_items': [(group == g).sum() for g in range(N_GROUPS)],
    'train_degree_min': [degree[group == g].min() for g in range(N_GROUPS)],
    'train_degree_max': [degree[group == g].max() for g in range(N_GROUPS)],
    'test_share': [np.isin(test.indices, np.flatnonzero(group == g)).mean() for g in range(N_GROUPS)],
})
display(group_info)

## CELL 4 — Tính chỉ số theo nhóm cho từng run

In [ ]:
test_group_counts = np.zeros((n_users, N_GROUPS), dtype=np.int32)
for u in range(n_users):
    t = test.indices[test.indptr[u]:test.indptr[u + 1]]
    if len(t):
        test_group_counts[u] = np.bincount(group[t], minlength=N_GROUPS)

def group_metrics(path, k):
    z = np.load(path, allow_pickle=False)
    users, items = z['user_ids'], z['item_ids'][:, :k]
    slots = np.zeros(N_GROUPS); hits = np.zeros(N_GROUPS)
    rec_sum = np.zeros(N_GROUPS); rec_n = np.zeros(N_GROUPS)
    for u, row in zip(users, items):
        tg = test_group_counts[u]
        if tg.sum() == 0:
            continue
        t = test.indices[test.indptr[u]:test.indptr[u + 1]]
        rg = group[row]
        hit = np.isin(row, t)
        slots += np.bincount(rg, minlength=N_GROUPS)
        hg = np.bincount(rg[hit], minlength=N_GROUPS)
        hits += hg
        m = tg > 0
        rec_sum[m] += hg[m] / tg[m]
        rec_n[m] += 1
    exposure = np.bincount(items.ravel(), minlength=n_items)
    return {
        'ExposureShare': slots / slots.sum(),
        'Precision': np.divide(hits, slots, out=np.zeros(N_GROUPS), where=slots > 0),
        'Recall': rec_sum / rec_n,
        'Coverage': np.array([(exposure[group == g] > 0).mean() for g in range(N_GROUPS)]),
        'n_eligible_users': rec_n,
        'Spearman': spearmanr(degree, exposure).correlation,
    }

per_run = []
for k in K_VALUES:
    for model, runs in EXPORTS.items():
        for seed, path in runs.items():
            m = group_metrics(path, k)
            for g in range(N_GROUPS):
                per_run.append({'model': model, 'seed': seed, 'K': k, 'group': GROUP_NAMES[g],
                                'ExposureShare': m['ExposureShare'][g], 'Precision': m['Precision'][g],
                                'Recall': m['Recall'][g], 'Coverage': m['Coverage'][g],
                                'n_eligible_users': int(m['n_eligible_users'][g]),
                                'DegreeExposureSpearman': m['Spearman']})
per_run = pd.DataFrame(per_run)
per_run.to_csv(OUT_DIR / 'popularity_groups_per_run.csv', index=False)
print('Saved', OUT_DIR / 'popularity_groups_per_run.csv')

## CELL 5 — Kiểm tra nhất quán với E2
G1 trùng với Head của E2, nên Recall G1 phải bằng Head Recall, tỷ lệ gợi ý G1 phải bằng HeadExposureShare, và Spearman phải bằng DegreeExposureSpearman.

In [ ]:
e2 = pd.read_csv(E2_SUMMARY_PATH)
e2_name = {'Most Popular': 'Most Popular', 'LightGCN': 'LightGCN', 'RLMRec-Con': 'RLMRec-Con'}
checks = []
for model, e2_model in e2_name.items():
    for k in K_VALUES:
        row = e2[(e2['model'] == e2_model) & (e2['K'] == k)].iloc[0]
        g1 = per_run[(per_run.model == model) & (per_run.K == k) & (per_run.group == 'G1')]
        checks.append({'model': model, 'K': k,
                       'HeadRecall_E2': row['HeadRecall_mean'], 'G1_Recall': g1['Recall'].mean(),
                       'HeadExposure_E2': row['HeadExposureShare_mean'], 'G1_Exposure': g1['ExposureShare'].mean(),
                       'Spearman_E2': row['DegreeExposureSpearman_mean'], 'Spearman': g1['DegreeExposureSpearman'].mean()})
checks = pd.DataFrame(checks)
display(checks.round(4))
assert np.allclose(checks['HeadRecall_E2'], checks['G1_Recall'], atol=1e-3), 'G1 Recall khác Head Recall của E2'
assert np.allclose(checks['Spearman_E2'], checks['Spearman'], atol=1e-3), 'Spearman khác E2'
print('Consistent with E2.')

## CELL 6 — Bảng mean ± SD theo nhóm

In [ ]:
metrics = ['ExposureShare', 'Precision', 'Recall', 'Coverage']
summary = (per_run.groupby(['K', 'model', 'group'], sort=False)[metrics]
           .agg(['mean', lambda x: x.std(ddof=1) if len(x) > 1 else np.nan]))
summary.columns = [f'{m}_{"mean" if s == "mean" else "sd"}' for m, s in summary.columns]
summary = summary.reset_index().merge(group_info[['group', 'train_degree_min', 'train_degree_max', 'test_share']], on='group')
for k in K_VALUES:
    out = summary[summary.K == k]
    out.to_csv(OUT_DIR / f'popularity_groups_k{k}.csv', index=False)
    print(f'K = {k}')
    display(out[['model', 'group', 'test_share', 'ExposureShare_mean', 'Recall_mean', 'Recall_sd',
                 'Precision_mean', 'Coverage_mean']].round(4))

## CELL 7 — Biểu đồ: tỷ lệ được gợi ý so với tỷ lệ test (K = 20)

In [ ]:
k = 20
plot_models = ['LightGCN', 'RLMRec-Con']
x = np.arange(N_GROUPS)
w = 0.27
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].bar(x - w, group_info['test_share'], w, label='Tỷ lệ test', color='lightgray')
for i, model in enumerate(plot_models):
    s = summary[(summary.K == k) & (summary.model == model)]
    axes[0].bar(x + i * w, s['ExposureShare_mean'], w, label=model)
    axes[1].bar(x + (i - 0.5) * w, s['Recall_mean'], w, yerr=s['Recall_sd'], label=model, capsize=3)
axes[0].set_title(f'Tỷ lệ được gợi ý theo nhóm (K={k})')
axes[1].set_title(f'Recall@{k} theo nhóm')
for ax in axes:
    ax.set_xticks(x); ax.set_xticklabels(GROUP_NAMES); ax.legend()
plt.tight_layout()
fig.savefig(OUT_DIR / 'popularity_groups_k20.png', dpi=200)
fig.savefig(OUT_DIR / 'popularity_groups_k20.pdf')
plt.show()